# SmokeWatch AI

Sistema intelligente per il rilevamento e la classificazione dell'opacità del fumo (Scala Ringelmann) emesso da navi e impianti industriali.

## Caratteristiche
- **Rilevamento**: YOLOv8 (Trainato su fumo e fuoco).
- **Analisi**: Calcolo dinamico Ringelmann 0-5 basato su luminanza ambientale.
- **Validazione**: Integrazione API OpenWeather (AQI) e Geocoding (Nominatim).

## Setup
1. Installa le dipendenze: `pip install -r requirements.txt`
2. Avvia il server: `python core/app.py`
3. Esegui un test batch: `python scripts/test_batch_api.py`

## Struttura
- `core/`: API Backend con FastAPI.
- `vision/`: Algoritmi di elaborazione immagini.
- `models/`: Pesi del modello YOLO.

# 🚢 Progetto SmokeWatch AI: Monitoraggio Ambientale Intelligente

Questo Notebook documenta il funzionamento e il workflow del sistema **SmokeWatch AI**, progettato per automatizzare il rilevamento delle emissioni di fumo e la classificazione della loro densità tramite la **Scala Ringelmann**.

---

## 🏗️ 1. Architettura del Sistema
L'applicazione è strutturata in modo modulare per separare la logica di visione artificiale dai servizi di arricchimento dati:

1. **Ingestion (Input):** Ricezione di immagini/video con metadati GPS e Heading.
2. **Detection (Vision):** Motore YOLOv8 per la localizzazione di fumo e fuoco.
3. **Quantificazione (Ringelmann):** Algoritmo di analisi della luminanza per il calcolo dell'opacità.
4. **Enrichment (Context):** Integrazione con API meteo (Open-Meteo) e qualità dell'aria (OpenAQ).
5. **Output:** Generazione di report JSON e archiviazione visiva dei test.

---

## 🔄 2. Workflow di Analisi

### A. Rilevamento tramite Computer Vision
Il sistema utilizza un modello **YOLOv8 Nano** ottimizzato per il riconoscimento di fumo e fiamme. 
* **Input:** Frame dell'immagine.
* **Output:** Bounding Box che delimita l'area di interesse (ROI).

### B. Calcolo Dinamico Scala Ringelmann
A differenza della semplice detection, SmokeWatch implementa una logica di analisi radiometrica:
1. **Campionamento Cielo:** Viene estratto un campione di luminanza dall'area immediatamente sopra il fumo per definire il "bianco di riferimento" ambientale.
2. **Analisi Opacità:** Viene calcolato il rapporto tra la luminanza del fumo e quella del cielo.
3. **Mappatura:** Il risultato viene convertito in un grado Ringelmann da **0 (trasparente)** a **5 (opaco totale)**.

### C. Analisi Ambientale e Geofencing
Se viene rilevata una criticità (Ringelmann > 1.0), il sistema interroga i servizi esterni:
* **Reverse Geocoding:** Identifica se l'emissione avviene in un'area portuale o industriale tramite Nominatim.
* **Dati Vento:** Recupera velocità e direzione del vento per prevedere la dispersione dei fumi.
* **AQI (Air Quality Index):** Verifica se i livelli di $NO_2$ e $SO_2$ nella zona sono già oltre la norma.

---

## 📂 3. Struttura dei Risultati
Ogni analisi genera automaticamente una sottocartella in `runs/detect/` organizzata come segue:
* `test_TIMESTAMP_NOMEFILE/`
    * 📄 `report.json`: Dati tecnici, meteo e coordinate GPS.
    * 🖼️ `detected_image.jpg`: Immagine annotata con i box di rilevamento.
    * 📷 `original_image.jpg`: Backup del file originale per fini legali.

---

## 🛠️ 4. Note Tecniche per il Deploy
* **Modello:** Caricato all'avvio nella classe `SmokeAnalyzer` per minimizzare la latenza.
* **API Keys:** Gestite tramite variabili d'ambiente o file `.env`.
* **Dipendenze:** Richiede `ultralytics`, `fastapi`, `opencv-python` e `requests`.

In [ ]:
# Cella di codice Jupyter
from vision.smoke_analysis import SmokeAnalyzer
import cv2
import matplotlib.pyplot as plt

# Inizializzazione
analyzer = SmokeAnalyzer(model_path="models/best.pt")

# Test rapido
img = cv2.imread("immagini_test/nave-inquinamento.jpg")
if img is not None:
    results = analyzer.analyze(img)
    print(f"Rilevamento completato! Grado Ringelmann: {results['ringelmann_avg']}")
    
    # Mostra l'immagine nel notebook
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.show()

In [ ]:
smokewatch/
├── core/                   # Logica di backend e API
│   ├── app.py              # Entry point del server FastAPI
│   └── main.py             # Orchestratore principale delle analisi
├── vision/                 # Intelligenza Artificiale e Image Processing
│   └── smoke_analysis.py   # Algoritmo YOLOv8 e calcolo Scala Ringelmann
├── service/                # Moduli per integrazioni esterne
│   └── environmental_services.py # API Meteo (Open-Meteo) e Qualità Aria (OpenAQ)
├── models/                 # Pesi dei modelli pre-addestrati
│   ├── best.pt             # Modello custom ottimizzato (da train-7)
│   └── yolov8n.pt          # Modello base YOLOv8 Nano
├── scripts/                # Utility per test e addestramento
│   ├── train_smokewatch.py # Script per il fine-tuning del modello
│   ├── test_batch_api.py   # Test automatizzato su più immagini via API
│   ├── test_massivo.py     # Stress test del sistema
│   └── test_model.py       # Validazione rapida del file .pt
├── data/                   # Gestione dei dati
│   ├── data.yaml           # Configurazione classi (0: fire, 1: smoke)
│   └── Fire and Smoke Dataset/ # Dataset locale per training/validation
├── .gitignore              # Esclusione file temporanei e pesanti (runs/, venv/)
├── requirements.txt        # Dipendenze del progetto (pip install -r)
└── README.md               # Manuale d'uso e documentazione tecnica